In [1]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
import os

In [11]:
park_shp = './dataset_structured/05_processed/standardized_layers/parks_with_quality_3857.shp'
res_shp = './dataset_structured/05_processed/standardized_layers/elderly_demand_3857.shp'

In [12]:
R = 2000
LAMBDA = 0.001

In [13]:
print("Loading data for Accessibility calculation...")
gdf_res = gpd.read_file(res_shp).to_crs(epsg=3857)
gdf_parks = gpd.read_file(park_shp).to_crs(epsg=3857)

res_coords = np.array(list(gdf_res.geometry.apply(lambda p: (p.x, p.y))))

park_coords = np.array(list(gdf_parks.geometry.centroid.apply(lambda p: (p.x, p.y))))
park_q = gdf_parks['Qj_index'].values

Loading data for Accessibility calculation...


In [14]:
print(gdf_res.head())

   id  population  elder_rati  elderly_de  demand_ind  \
0  64   15.257054    0.078786    1.202037    0.023637   
1  65   20.436389    0.078786    1.610094    0.031784   
2  66   17.545509    0.078786    1.382334    0.027237   
3  67   17.431071    0.078786    1.373318    0.027057   
4  68   40.303839    0.078786    3.175364    0.063033   

                       geometry  
0  POINT (13517450 3673468.474)  
1  POINT (13517550 3673468.474)  
2  POINT (13517650 3673468.474)  
3  POINT (13517750 3673468.474)  
4  POINT (13517850 3673468.474)  


In [ ]:
print(f"Building spatial tree and calculating Ai (Radius={R}m)...")
tree = cKDTree(park_coords)

indices = tree.query_ball_point(res_coords, r=R)

ai_results = []
for i, neighbors in enumerate(indices):
    if not neighbors:
        ai_results.append(0)
        continue
    
    
    dists = np.linalg.norm(park_coords[neighbors] - res_coords[i], axis=1)
    
    # Ai = sum( Qj * exp(-lambda * dij) )
    step_scores = park_q[neighbors] * np.exp(-LAMBDA * dists)
    ai_results.append(np.sum(step_scores))

gdf_res['Ai_access'] = ai_results

# Weighted_A = Ai_access * elderly_demand
gdf_res['Weighted_A'] = gdf_res['Ai_access'] * gdf_res['elderly_de']

Building spatial tree and calculating Ai (Radius=2000m)...


In [16]:
output_path = './dataset_structured/05_processed/standardized_layers/accessibility_final_grid.shp'
gdf_res.to_file(output_path, encoding='utf-8')

print(f"Success! Final accessibility map saved to: {output_path}")

Success! Final accessibility map saved to: ./dataset_structured/05_processed/standardized_layers/accessibility_final_grid.shp
